<a href="https://colab.research.google.com/github/rizanatlanta-dsproj/Mental-chatbot/blob/main/mentalchatbot_iu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai gradio textblob

In [2]:
import os
import gradio as gr
from textblob import TextBlob
from google import genai
from google.genai import types
from google.colab import userdata

In [3]:
# Initializinggg Gemini Client via Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except Exception:
    print("⚠️ Add your GEMINI_API_KEY to the Colab Secrets tab (key icon)!")

⚠️ Add your GEMINI_API_KEY to the Colab Secrets tab (key icon)!


In [12]:
import os
import gradio as gr
from textblob import TextBlob
from google import genai
from google.genai import types
from google.colab import userdata

# Initialize Gemini Client via Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except Exception:
    print("⚠️ Add your GEMINI_API_KEY to the Colab Secrets tab (key icon)!")

class PortfolioMentalHealthBot:
    def __init__(self):
        self.states = ["1. ASSESSMENT", "2. COGNITIVE REFRAMING", "3. FINAL REFLECTION"]
        self.current_state_idx = 0
        self.chat_history = []

    def extract_metrics(self, text: str):
        """Algorithmic parsing of user input to show system intelligence."""
        blob = TextBlob(text)
        polarity = round(blob.sentiment.polarity, 2) # -1.0 to +1.0

        # Simple heuristic emotion detection rule-engine
        text_lower = text.lower()
        detected_emotion = "Neutral"

        joy_words = ["happy", "good", "better", "glad", "excited", "relieved"]
        sad_words = ["sad", "down", "depressed", "hurt", "crying", "lonely", "heavy"]
        angry_words = ["mad", "angry", "furious", "hate", "annoyed", "frustrated"]
        fear_words = ["scared", "anxious", "panic", "worried", "afraid", "terrified"]

        if any(w in text_lower for w in sad_words) or polarity < -0.3:
            detected_emotion = "Sadness 😢"
        if any(w in text_lower for w in fear_words):
            detected_emotion = "Anxiety/Fear 😰"
        if any(w in text_lower for w in angry_words):
            detected_emotion = "Anger 😠"
        if any(w in text_lower for w in joy_words) or polarity > 0.3:
            detected_emotion = "Joy/Relief 😊"

        return polarity, detected_emotion

    def check_safety(self, text: str) -> bool:
        crisis_terms = ["suicide", "harm", "kill myself", "end my life", "cut myself"]
        return any(term in text.lower() for term in crisis_terms)

    def process_turn(self, user_message: str, current_history):
        if not user_message.strip():
            return "", current_history, self.states[self.current_state_idx], 0, "Neutral"

        # 1. Safety Gate Check
        if self.check_safety(user_message):
            response = "⚠️ [SAFETY TRIGGER] It sounds like you might be in immediate distress. As an AI prototype, I am not equipped to handle crises. Please reach out to a trusted professional or contact your local emergency services right away."
            current_history.append((user_message, response))
            return "", current_history, self.states[self.current_state_idx], -1.0, "CRITICAL RISK ⚠️"

        # 2. Extract telemetry data
        polarity, emotion = self.extract_metrics(user_message)
        current_stage = self.states[self.current_state_idx]

        # 3. Formulate structural system instructions
        system_instruction = f"""
        You are a highly constrained, empathetic AI therapeutic framework designed for an Intershippp argh.
        Your task is to help the user complete the current stage of cognitive reframing.

        CURRENT STAGE: {current_stage}
        - 1. ASSESSMENT: Validate their feeling warmly, and ask them to name the specific negative thought loop.
        - 2. COGNITIVE REFRAMING: Guide them to look at the facts and find one rational alternative viewpoint.
        - 3. FINAL REFLECTION: Ask them what they can take away from this exercise moving forward.

        REAL-TIME BIO-METRICS LOGGED BY THE SYSTEM:
        - User Sentiment Polarity: {polarity} (-1 is highly negative, +1 is highly positive)
        - Core Detected Emotion: {emotion}

        Tone constraint: Be deeply empathetic, brief (under 3 sentences), and do not use medical diagnostic labels.
        """

        # Format history properly for native GenAI SDK structure
        self.chat_history.append({"role": "user", "parts": [user_message]})
        sdk_contents = [types.Content(role=h["role"], parts=[types.Part.from_text(text=h["parts"][0])]) for h in self.chat_history]

        try:
            api_response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=sdk_contents,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.6,
                )
            )
            bot_reply = api_response.text
            self.chat_history.append({"role": "model", "parts": [bot_reply]})

            # Step forward through the state machine programmatically
            if self.current_state_idx < len(self.states) - 1:
                self.current_state_idx += 1

        except Exception as e:
            bot_reply = f"System Link Error: {str(e)}"

        current_history.append((user_message, bot_reply))
        return "", current_history, self.states[self.current_state_idx], polarity, emotion

    def reset_bot(self):
        self.current_state_idx = 0
        self.chat_history = []
        return [], self.states[0], 0.0, "Neutral"

# Instantiate engine
engine = PortfolioMentalHealthBot()

#hope it doesn't blow my laptop up

⚠️ Add your GEMINI_API_KEY to the Colab Secrets tab (key icon)!


In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 MindFlow AI — Interactive Portfolio Sandbox")
    gr.Markdown("---")

    with gr.Row():
        # Left Panel: Telemetry Metrics Dashboard (The Admissions Highlight)
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Real-Time Backend Telemetry")
            state_box = gr.Textbox(label="Active State Machine Stage", value=engine.states[0], interactive=False)
            sentiment_bar = gr.Slider(label="Sentiment Polarity (-1.0 to +1.0)", minimum=-1.0, maximum=1.0, value=0.0, interactive=False)
            emotion_box = gr.Textbox(label="Detected Categorical Emotion", value="Neutral", interactive=False)
            reset_btn = gr.Button("🔄 Restart Conversation Sequence", variant="stop")

        # Right Panel: Live Chat
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="MindFlow Clinical AI Core Engine Interface")
            msg_input = gr.Textbox(label="Type your response here...", placeholder="e.g., I'm feeling incredibly stressed about exams...", lines=1)

    # Link input triggers to processing logic
    msg_input.submit(
        fn=engine.process_turn,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot, state_box, sentiment_bar, emotion_box]
    )

    reset_btn.click(
        fn=engine.reset_bot,
        outputs=[chatbot, state_box, sentiment_bar, emotion_box]
    )

# Launch with share=True to generate a temporary public link if you want to test on your phone!
demo.launch(debug=True, inline=True)

/tmp/ipykernel_3187/3029212401.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bb112cf8f9cb7bdf8f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2293, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2043, in postprocess_data
    prediction_value = await anyio.to_thread.run_syn